In [1]:
# =========================================================
# 0. IMPORT LIBRARIES
# =========================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


C:\Users\vivek\anaconda4\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv("Telco-Customer-Churn.csv")

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(" ", np.nan))

print("BEFORE DROPPING OR IMPUTING NaN IN TotalCharges:")
print(df['TotalCharges'].isna().sum())



BEFORE DROPPING OR IMPUTING NaN IN TotalCharges:
11


In [3]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({1:"Yes",0:"No"})
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
X = df.drop(['customerID','Churn'], axis=1)
y = df['Churn'].map({'Yes':1, 'No':0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)


Training set size: (5634, 19)
Test set size: (1409, 19)


In [5]:
numeric_cols = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


Numeric columns: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [6]:
# BEFORE
print("\nMissing values BEFORE imputation:")
print(X_train[numeric_cols].isna().sum())
print(X_train[categorical_cols].isna().sum())

# IMPUTE
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

# Numeric: median
for col in numeric_cols:
    median = X_train_imputed[col].median()
    X_train_imputed[col].fillna(median, inplace=True)
    X_test_imputed[col].fillna(median, inplace=True)   # test uses train median

# Categorical: mode
for col in categorical_cols:
    mode = X_train_imputed[col].mode()[0]
    X_train_imputed[col].fillna(mode, inplace=True)
    X_test_imputed[col].fillna(mode, inplace=True)

# AFTER
print("\nMissing values AFTER imputation:")
print(X_train_imputed[numeric_cols].isna().sum())
print(X_train_imputed[categorical_cols].isna().sum())



Missing values BEFORE imputation:
tenure            0
MonthlyCharges    0
TotalCharges      8
dtype: int64
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
dtype: int64

Missing values AFTER imputation:
tenure            0
MonthlyCharges    0
TotalCharges      0
dtype: int64
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
dtype: int64


C:\Users\vivek\AppData\Local\Temp\ipykernel_27864\3302375929.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train_imputed[col].fillna(median, inplace=True)
C:\Users\vivek\AppData\Local\Temp\ipykernel_27864\3302375929.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For 

In [7]:
X_train_clipped = X_train_imputed.copy()
X_test_clipped = X_test_imputed.copy()

continuous_cols = [col for col in numeric_cols if X_train[col].nunique() > 10]

print("\nClipping only continuous numeric columns:", continuous_cols)

for col in continuous_cols:
    Q1 = X_train_clipped[col].quantile(0.25)
    Q3 = X_train_clipped[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X_train_clipped[col] = X_train_clipped[col].clip(lower, upper)
    X_test_clipped[col] = X_test_clipped[col].clip(lower, upper)



Clipping only continuous numeric columns: ['tenure', 'MonthlyCharges', 'TotalCharges']


In [8]:
binary_cols = [col for col in categorical_cols if X_train_clipped[col].nunique()==2]
multi_cols  = [col for col in categorical_cols if X_train_clipped[col].nunique()>2]

print("\nBinary categorical columns:", binary_cols)
print("Multi-category columns:", multi_cols)



Binary categorical columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multi-category columns: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [9]:
print("\n=== BEFORE LABEL ENCODING (sample) ===")
print(X_train_clipped[binary_cols].head())

le = LabelEncoder()
X_train_encoded = X_train_clipped.copy()
X_test_encoded = X_test_clipped.copy()

for col in binary_cols:
    X_train_encoded[col] = le.fit_transform(X_train_clipped[col])
    X_test_encoded[col]  = le.transform(X_test_clipped[col])

print("\n=== AFTER LABEL ENCODING (sample) ===")
print(X_train_encoded[binary_cols].head())



=== BEFORE LABEL ENCODING (sample) ===
      gender SeniorCitizen Partner Dependents PhoneService PaperlessBilling
3738    Male            No      No         No           No               No
3151    Male            No     Yes        Yes          Yes               No
4860    Male            No     Yes        Yes           No               No
3867  Female            No     Yes         No          Yes              Yes
3810    Male            No     Yes        Yes          Yes               No

=== AFTER LABEL ENCODING (sample) ===
      gender  SeniorCitizen  Partner  Dependents  PhoneService  \
3738       1              0        0           0             0   
3151       1              0        1           1             1   
4860       1              0        1           1             0   
3867       0              0        1           0             1   
3810       1              0        1           1             1   

      PaperlessBilling  
3738                 0  
3151              

In [10]:
print("\n=== BEFORE ONE-HOT ENCODING (columns):")
print(multi_cols)

ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
ohe_train = ohe.fit_transform(X_train_encoded[multi_cols])
ohe_test  = ohe.transform(X_test_encoded[multi_cols])

ohe_cols = ohe.get_feature_names_out(multi_cols)

X_train_ohe = pd.concat([
    X_train_encoded.drop(columns=multi_cols).reset_index(drop=True),
    pd.DataFrame(ohe_train, columns=ohe_cols)
], axis=1)

X_test_ohe = pd.concat([
    X_test_encoded.drop(columns=multi_cols).reset_index(drop=True),
    pd.DataFrame(ohe_test, columns=ohe_cols)
], axis=1)

print("\n=== AFTER ONE-HOT ENCODING (new columns):")
print(ohe_cols)



=== BEFORE ONE-HOT ENCODING (columns):
['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

=== AFTER ONE-HOT ENCODING (new columns):
['MultipleLines_No phone service' 'MultipleLines_Yes'
 'InternetService_Fiber optic' 'InternetService_No'
 'OnlineSecurity_No internet service' 'OnlineSecurity_Yes'
 'OnlineBackup_No internet service' 'OnlineBackup_Yes'
 'DeviceProtection_No internet service' 'DeviceProtection_Yes'
 'TechSupport_No internet service' 'TechSupport_Yes'
 'StreamingTV_No internet service' 'StreamingTV_Yes'
 'StreamingMovies_No internet service' 'StreamingMovies_Yes'
 'Contract_One year' 'Contract_Two year'
 'PaymentMethod_Credit card (automatic)' 'PaymentMethod_Electronic check'
 'PaymentMethod_Mailed check']


In [11]:
scaler = StandardScaler()

print("\n=== BEFORE SCALING (numeric stats) ===")
print(X_train_ohe[numeric_cols].describe().T)

X_train_final = X_train_ohe.copy()
X_test_final = X_test_ohe.copy()

X_train_final[numeric_cols] = scaler.fit_transform(X_train_ohe[numeric_cols])
X_test_final[numeric_cols] = scaler.transform(X_test_ohe[numeric_cols])

print("\n=== AFTER SCALING (numeric stats) ===")
print(X_train_final[numeric_cols].describe().T)



=== BEFORE SCALING (numeric stats) ===
                 count         mean          std    min       25%       50%  \
tenure          5634.0    32.485091    24.568744   0.00    9.0000    29.000   
MonthlyCharges  5634.0    64.929961    30.138105  18.40   35.6625    70.500   
TotalCharges    5634.0  2301.319950  2277.808844  18.85  408.8500  1398.125   

                     75%      max  
tenure            55.000    72.00  
MonthlyCharges    90.000   118.75  
TotalCharges    3835.825  8684.80  

=== AFTER SCALING (numeric stats) ===
                 count          mean       std       min       25%       50%  \
tenure          5634.0 -1.008935e-17  1.000089 -1.322329 -0.955978 -0.141863   
MonthlyCharges  5634.0 -2.402527e-16  1.000089 -1.544028 -0.971198  0.184834   
TotalCharges    5634.0 -2.017871e-17  1.000089 -1.002135 -0.830903 -0.396554   

                     75%       max  
tenure          0.916486  1.608483  
MonthlyCharges  0.831912  1.785939  
TotalCharges    0.673736  2.

In [12]:
print("\nBEFORE SMOTE:")
print("Class distribution:", y_train.value_counts())

sm = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = sm.fit_resample(X_train_final, y_train)

print("\nAFTER SMOTE:")
print("Class distribution:", y_train_balanced.value_counts())



BEFORE SMOTE:
Class distribution: Churn
0    4139
1    1495
Name: count, dtype: int64

AFTER SMOTE:
Class distribution: Churn
0    4139
1    4139
Name: count, dtype: int64


In [13]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000),
    'DecisionTree': DecisionTreeClassifier(),
    'RandomForest': RandomForestClassifier(),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss')
}

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train_balanced, y_train_balanced, cv=5, scoring='accuracy')
    cv_results[name] = scores.mean()

print("\n=== BASELINE CV RESULTS ===")
print(cv_results)



=== BASELINE CV RESULTS ===
{'LogisticRegression': 0.7705940861392063, 'DecisionTree': 0.7818379380299778, 'RandomForest': 0.8514200125516297, 'XGBoost': 0.8360846213348513}


In [14]:
best_model_name = max(cv_results, key=cv_results.get)
best_model = models[best_model_name]

best_model.fit(X_train_balanced, y_train_balanced)
y_pred = best_model.predict(X_test_final)

print("\n=== FINAL TEST SET RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



=== FINAL TEST SET RESULTS ===
Accuracy: 0.7714691270404542
              precision    recall  f1-score   support

           0       0.86      0.83      0.84      1035
           1       0.56      0.61      0.59       374

    accuracy                           0.77      1409
   macro avg       0.71      0.72      0.71      1409
weighted avg       0.78      0.77      0.77      1409

